# Ejercicio 9: Uso de la API de Google Gemini

En este ejercicio vamos a aprender a utilizar la API de OpenAI

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [1]:
import sys
!{sys.executable} -m pip install google-genai
!pip install ipywidgets
!{sys.executable} -m pip install scikit-learn
!{sys.executable} -m pip install sentence-transformers


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from google import genai

# La API key se lee desde un archivo de texto local, NUNCA escrita directamente en el código
with open("api_key.txt", "r") as f:
    api_key = f.read().strip()

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="¿Cuál es la capital de Francia?",
)

print(response.text)



La capital de Francia es **París**.


## 2. Retrieval

### 2.1 Cargo el corpus de 20 News Groups

In [4]:
from sklearn.datasets import fetch_20newsgroups

# Cargo un subconjunto para mayor velocidad
newsgroups = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))

# Tomo los primeros 500 documentos
documents = newsgroups.data[:500]
print(f"Total de documentos cargados: {len(documents)}")
print(f"\nEjemplo de documento:\n{documents[0][:300]}...")

Total de documentos cargados: 500

Ejemplo de documento:
I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I k...


### 2.2 Transformo a embeddings

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Cargamos el modelo local
model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Iniciando el procesamiento local de {len(documents)} documentos...")

# Generar embeddings
embeddings_matrix = model.encode(
    documents, 
    show_progress_bar=True, 
    convert_to_numpy=True
)

# 4. Ver el resultado final
print(f"\n✅ ¡Proceso completado!")
print(f"Shape de la matriz de embeddings: {embeddings_matrix.shape}")


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3432.76it/s]


Iniciando el procesamiento local de 500 documentos...


Batches: 100%|██████████| 16/16 [00:13<00:00,  1.23it/s]


✅ ¡Proceso completado!
Shape de la matriz de embeddings: (500, 384)


### 2.3 Creo una query y hago la búsqueda

In [7]:
# 1. Defino una query de búsqueda
query = "NASA space shuttle launch mission"

# 2. Obtengo el embedding usando el modelo local
query_embedding = model.encode(query)

print(f"Query: '{query}'")
print(f"Shape del embedding de la query: {query_embedding.shape}")

Query: 'NASA space shuttle launch mission'
Shape del embedding de la query: (384,)


Obtengo los 5 documentos más similares a mi query

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculo similitud coseno entre la query y todos los documentos
similarities = cosine_similarity([query_embedding], embeddings_matrix)[0]

# Obtengo los índices de los 5 más similares
top_5_indices = np.argsort(similarities)[::-1][:5]

print(f"Top 5 documentos más similares a: '{query}'\n")
print("="*60)
for rank, idx in enumerate(top_5_indices, 1):
    print(f"\n🔹 Rank {rank} | Índice: {idx} | Similitud: {similarities[idx]:.4f}")
    print("-"*60)
    print(documents[idx][:400])
    print("...")

Top 5 documentos más similares a: 'NASA space shuttle launch mission'


🔹 Rank 1 | Índice: 153 | Similitud: 0.5825
------------------------------------------------------------
Archive-name: space/schedule
Last-modified: $Date: 93/04/01 14:39:23 $

SPACE SHUTTLE ANSWERS, LAUNCH SCHEDULES, TV COVERAGE

    SHUTTLE LAUNCHINGS AND LANDINGS; SCHEDULES AND HOW TO SEE THEM

    Shuttle operations are discussed in the Usenet group sci.space.shuttle,
    and Ken Hollis (gandalf@pro-electric.cts.com) posts a compressed version
    of the shuttle manifest (launch dates and other i
...

🔹 Rank 2 | Índice: 282 | Similitud: 0.3876
------------------------------------------------------------

DFW was designed with the STS in mind (which really mean very little).  Much of
their early PR material had scenes with a shuttle landing and two or three
others pulled up to gates.  I guess they were trying to stress how advanced the
airport was.

For Dallas types:  Imagine the fit Grapevine and Irving would be

In [9]:
# Resumen de resultados en tabla
print("\nResumen de similitudes:")
print(f"{'Rank':<6} {'Índice':<8} {'Similitud':<12} {'Inicio del documento'}")
print("="*70)
for rank, idx in enumerate(top_5_indices, 1):
    preview = documents[idx][:60].replace('\n', ' ')
    print(f"{rank:<6} {idx:<8} {similarities[idx]:<12.4f} {preview}...")


Resumen de similitudes:
Rank   Índice   Similitud    Inicio del documento
1      153      0.5825       Archive-name: space/schedule Last-modified: $Date: 93/04/01 ...
2      282      0.3876        DFW was designed with the STS in mind (which really mean ve...
3      467      0.3515       News-Software: UReply 3.1 X-X-From: Wingert@VNET.IBM.com (Br...
4      232      0.3278         SPECIFIC: Basically to be able to do the things the big da...
5      322      0.3195        Actually, the reboost will probably be done last, so that t...
